In [ ]:
pip install pandas numpy scikit-learn scipy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# 載入 ARFF 檔
data, meta = arff.loadarff('phishing_dataset.arff')
df = pd.DataFrame(data)

# 如果欄位被讀成 bytes 格式，轉成整數
for col in df.columns:
    df[col] = df[col].astype(int)

In [ ]:
from google.colab import files
from scipy.io import arff
import pandas as pd

# 1. 彈出上傳視窗，選取你的 arff 檔案
uploaded = files.upload()

# 2. 取得上傳的檔名並載入
filename = list(uploaded.keys())[0]
data, meta = arff.loadarff(filename)

# 3. 轉成 Pandas DataFrame
df = pd.DataFrame(data)

# 若字串資料帶有 b' 前綴（bytes 格式），將其解碼轉成普通字串或數字
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.decode('utf-8')

print("載入成功！前 5 筆資料：")
print(df.head())


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# 1. 資料格式整理 (將文字/bytes 轉為數值)
# --------------------------------------------------
for col in df.columns:
    # 若為 bytes 類型，解碼為字串
    if df[col].dtype == object:
        df[col] = df[col].apply(
            lambda x: x.decode("utf-8") if isinstance(x, bytes) else x
        )
    # 將所有特徵轉為整數
    df[col] = df[col].astype(int)

# 2. 將目標欄位 Result 的 -1 (釣魚) 轉為 0，1 (合法) 保持為 1
# （部分 ARFF 欄位名稱可能是大寫或小寫，這裡做自動修正）
target_col = [c for c in df.columns if c.lower() == "result"][0]
df[target_col] = df[target_col].map({-1: 0, 1: 1})

# 3. 分離特徵 (X) 與標籤 (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# --------------------------------------------------
# 2. 切分訓練集與測試集 (80% 訓練, 20% 測試)
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ 資料處理解析完成！")
print(f"訓練集數量：{len(X_train)} 筆 | 測試集數量：{len(X_test)} 筆\n")

# --------------------------------------------------
# 3. 訓練隨機森林模型 (Random Forest)
# --------------------------------------------------
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --------------------------------------------------
# 4. 評估模型效果
# --------------------------------------------------
y_pred = model.predict(X_test)

print("=== 📊 分類結果評估報告 ===")
print(
    classification_report(
        y_test, y_pred, target_names=["Phishing (釣魚)", "Legitimate (合法)"]
    )
)

print("=== 🧩 混淆矩陣 (Confusion Matrix) ===")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from google.colab import files

# 1. 將處理完成的 DataFrame (df) 儲存為 CSV 檔
df.to_csv("phishing_processed.csv", index=False)

# 2. 自動跳出下載視窗，將檔案下載到你的電腦
files.download("phishing_processed.csv")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 從 .csv 檔案讀取資料
df_csv = pd.read_csv("phishing_dataset.csv")

# 2. 分離特徵 (X) 與目標變數 (y)
target_col = [c for c in df_csv.columns if c.lower() == "result"][0]
X = df_csv.drop(columns=[target_col])
y = df_csv[target_col]

# 3. 切分訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 特徵標準化 (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. 使用標準化後的資料訓練模型與分析
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 6. 評估分析結果
y_pred = model.predict(X_test_scaled)
print("=== 📊 從 .csv 讀取並經過 Standardize 後的分析結果 ===")
print(
    classification_report(
        y_test, y_pred, target_names=["Phishing (釣魚)", "Legitimate (合法)"]
    )
)

In [ ]:
from google.colab import files

# 將清理後的資料存成 .csv
df.to_csv("phishing_dataset.csv", index=False)

# 下載 csv 檔案到電腦（要給夥伴的檔案）
files.download("phishing_dataset.csv")
print("✅ .csv 檔案已成功導出！")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 讀取匯出的 CSV 檔案
df = pd.read_csv("phishing_dataset.csv")

# 2. 分離特徵 (X) 與標籤 (y)
X = df.drop(columns=["Result"])
y = df["Result"]

# 3. 切分訓練集 (80%) 與測試集 (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 特徵標準化 (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. 訓練隨機森林模型
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 6. 模型預測與評估
y_pred = model.predict(X_test_scaled)

print("=== 📊 從 .csv 讀取並完成 Standardize 後的分析結果 ===")
print(
    classification_report(
        y_test, y_pred, target_names=["Phishing (0)", "Legitimate (1)"]
    )
)
print("=== 🧩 混淆矩陣 (Confusion Matrix) ===")
print(confusion_matrix(y_test, y_pred))